# Piloto: fibras e derivadas quadráticas
Uma família H=0xa2000; limite do solver 300 s, mais preparação e auditoria. CPU, sem A100.
SAT não certifica bentness. UNSAT só conta após auditoria do modelo e verificação DRAT.
Execute em um ambiente separado; este notebook não altera nem interrompe o lote anterior.


In [ ]:
from pathlib import Path
import subprocess, sys
BASE=Path('/content/hrsbf_derivative_pilot')
BASE.mkdir(exist_ok=True)
def command(args,cwd=None):
    with subprocess.Popen([str(a) for a in args],cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
        for line in p.stdout: print(line,end='',flush=True)
        status=p.wait()
        if status: raise subprocess.CalledProcessError(status,args)
PROJECT=BASE/'project'
if not PROJECT.exists(): command(['git','clone','https://github.com/contatofilipevono-rgb/hrsbf-bent-verification.git',PROJECT])
command(['git','fetch','origin','pesquisa-t32-2026-10-04'],PROJECT)
command(['git','checkout','47360266fc38354bd46c3de0b98f67567fb13944'],PROJECT)
for name,url in [('cadical','https://github.com/arminbiere/cadical.git'),('drat-trim','https://github.com/marijnheule/drat-trim.git')]:
    path=BASE/name
    if not path.exists(): command(['git','clone','--depth','1',url,path])
    print(name,subprocess.check_output(['git','rev-parse','HEAD'],cwd=path,text=True).strip())
if not (BASE/'cadical/build/cadical').exists():
    command(['./configure'],BASE/'cadical')
    command(['make','-j2'],BASE/'cadical')
if not (BASE/'drat-trim/drat-trim').exists(): command(['make','drat-trim'],BASE/'drat-trim')
command([sys.executable,PROJECT/'pesquisa_t32/derivative_pilot.py','--base',BASE,'--h','0xa2000','--seconds','300'])


In [ ]:
from pathlib import Path
import json
from google.colab import files
out=Path('/content/hrsbf_derivative_pilot/resultados_derivadas_piloto')
summary=out/'summary.json'
if summary.exists(): print(json.dumps(json.loads(summary.read_text()),indent=2))
archive=out/'resultado_derivadas_piloto.zip'
if archive.exists(): files.download(str(archive))
else: print('ZIP ainda não disponível; aguarde o piloto.')
